In [ ]:
data = data.drop(columns=['hora_inicio', 'alcaldia_catalogo', 'municipio_hecho'])
print(f"Columnas eliminadas: hora_inicio, alcaldia_catalogo, municipio_hecho")
print(f"Quedan: {data.shape[1]} columnas")

Columnas eliminadas: hora_inicio, alcaldia_catalogo, municipio_hecho
Quedan: 18 columnas


In [ ]:
meses_eng = {'August':'Agosto','July':'Julio','March':'Marzo',
             'April':'Abril','February':'Febrero','June':'Junio','May':'Mayo'}
data['mes_inicio'] = data['mes_inicio'].replace(meses_eng)
data['mes_hecho'] = data['mes_hecho'].replace(meses_eng)
print(f"Meses en inglés corregidos")
print(f"mes_inicio únicos: {data['mes_inicio'].nunique()}")
print(f"mes_hecho únicos: {data['mes_hecho'].nunique()}")

Meses en inglés corregidos
mes_inicio únicos: 12
mes_hecho únicos: 12


In [ ]:
#Quinta celda — marcar datos no confiables:

data['fecha_sospechosa'] = (
    data['anio_hecho'].isnull() |
    (data['anio_hecho'] < 2016) |
    (data['anio_hecho'] > 2025)
)
data['hora_hecho_valida'] = (
    data['hora_hecho'].notna() &
    (data['hora_hecho'] != '00:00:00') &
    (data['hora_hecho'] != 'nan')
)
print(f"Fechas sospechosas: {data['fecha_sospechosa'].sum():,}")
print(f"Horas hecho válidas: {data['hora_hecho_valida'].sum():,}")

Fechas sospechosas: 33,158
Horas hecho válidas: 2,069,920


In [ ]:
#Sexta celda — convertir 'nan' strings a nulos reales:
for col in data.columns:
    if data[col].dtype == object or str(data[col].dtype) == 'str':
        data[col] = data[col].replace('nan', np.nan)

print(f"Strings 'nan' convertidos a nulos reales")
print(f"Nulos en fiscalia: {data['fiscalia'].isnull().sum()}")
print(f"Nulos en competencia: {data['competencia'].isnull().sum():,}")

Strings 'nan' convertidos a nulos reales
Nulos en fiscalia: 2
Nulos en competencia: 1,064,018


In [ ]:
#Séptima celda — verificación final y guardar:
print("Revisión final de datos de la fase 1 de limpieza")
print(f"\nRegistros: {len(data):,}")
print(f"Columnas: {data.shape[1]} → {list(data.columns)}")
print(f"\nNulos por columna:")
for col in data.columns:
    n = data[col].isnull().sum()
    if n > 0:
        print(f"  {col:25s} → {n:,} ({n/len(data)*100:.1f}%)")
    else:
        print(f"  {col:25s} → ✓ completa")

data.to_csv("carpetasFGJ_fase1.csv", index=False)
print(f"\nGuardado: carpetasFGJ_fase1.csv")

VERIFICACIÓN FINAL - FASE 1

Registros: 2,098,743
Columnas: 20 → ['anio_inicio', 'mes_inicio', 'fecha_inicio', 'anio_hecho', 'mes_hecho', 'fecha_hecho', 'hora_hecho', 'delito', 'categoria_delito', 'competencia', 'fiscalia', 'agencia', 'unidad_investigacion', 'colonia_hecho', 'colonia_catalogo', 'alcaldia_hecho', 'latitud', 'longitud', 'fecha_sospechosa', 'hora_hecho_valida']

Nulos por columna:
  anio_inicio               → ✓ completa
  mes_inicio                → ✓ completa
  fecha_inicio              → 3 (0.0%)
  anio_hecho                → 559 (0.0%)
  mes_hecho                 → 559 (0.0%)
  fecha_hecho               → 560 (0.0%)
  hora_hecho                → 887 (0.0%)
  delito                    → ✓ completa
  categoria_delito          → ✓ completa
  competencia               → 1,064,018 (50.7%)
  fiscalia                  → 2 (0.0%)
  agencia                   → ✓ completa
  unidad_investigacion      → 978 (0.0%)
  colonia_hecho             → 102,124 (4.9%)
  colonia_catalogo   

# Hasta aqui llega ROCHA en el eda

# INICIA FASE 1

In [ ]:
"""
=============================================================================
Urban Crime CDMX — Etapa 5.1: CORRECCIONES al mapeo de taxonomía
=============================================================================
Aplica fixes puntuales identificados en la revisión manual.
Ejecutar DESPUÉS del script principal de taxonomía.
=============================================================================
"""

import pandas as pd

# ============================================================================
# Cargar mapeo y dataset
# ============================================================================
mapeo = pd.read_csv("taxonomia_delitos_mapeo.csv")
df = pd.read_csv("carpetasFGJ_fase2.csv")

print(f"Mapeo: {len(mapeo)} filas")
print(f"Dataset: {len(df):,} registros")

# ============================================================================
# Definir correcciones
# ============================================================================
correcciones = {
    # --- Errores TF-IDF ---
    # 1. TENTATIVA DE SUICIDIO: FEMINICIDIO → PERDIDA DE LA VIDA
    "TENTATIVA DE SUICIDIO": "PERDIDA DE LA VIDA",
    
    # 2. POSESION DE VEHICULO ROBADO: NARCOMENUDEO → ROBO SIN VIOLENCIA
    "POSESION DE VEHICULO ROBADO": "ROBO SIN VIOLENCIA",
    
    # 3-4. INHUMACIONES: LESIONES INTENCIONALES → OTROS DELITOS
    "INHUMACION, EXHUMACION Y RESPETO A LOS CADAVERES O RESTOS HUMANOS": "OTROS DELITOS",
    "INHUMACIONES Y/O EXHUMACIONES": "OTROS DELITOS",
    
    # 5. USURPACION DE FUNCIONES: se queda en DELITOS DE SERVIDORES PUBLICOS (confirmado)
    # 6. USO INDEBIDO DE CONDECORACIONES: se queda en DELITOS PROFESIONALES (confirmado)
    
    # --- Errores por regla (keyword "VIOLACION" mal capturado) ---
    # 7. VIOLACION DE CORRESPONDENCIA: DELITOS SEXUALES → OTROS DELITOS
    "VIOLACION DE CORRESPONDENCIA": "OTROS DELITOS",
    
    # 8. VIOLACION A LOS DERECHOS HUMANOS: DELITOS SEXUALES → DISCRIMINACION Y DERECHOS HUMANOS
    "VIOLACION A LOS DERECHOS HUMANOS": "DISCRIMINACION Y DERECHOS HUMANOS",
    
    # 9. SECUESTRO EXPRESS: ROBO SIN VIOLENCIA → ROBO CON VIOLENCIA
    "SECUESTRO EXPRESS (PARA COMETER ROBO O EXTORSIÓN)": "ROBO CON VIOLENCIA",
    
    # 10. Delitos compuestos con violación + robo: ROBO SIN VIOLENCIA → DELITOS SEXUALES
    "VIOLACION EQUIPARADA Y ROBO DE VEHICULO": "DELITOS SEXUALES",
    "VIOLACION Y ROBO DE VEHICULO": "DELITOS SEXUALES",
}

# ============================================================================
# Aplicar correcciones al mapeo
# ============================================================================
print(f"\n{'='*80}")
print(f"APLICANDO CORRECCIONES:")
print(f"{'='*80}")

corregidos = 0
for delito_original, nueva_macro in correcciones.items():
    mask = mapeo['delito_original'] == delito_original
    if mask.sum() == 0:
        print(f"  ⚠️  NO ENCONTRADO: '{delito_original}'")
        continue
    
    vieja_macro = mapeo.loc[mask, 'macro_categoria'].values[0]
    conteo = mapeo.loc[mask, 'conteo'].values[0]
    mapeo.loc[mask, 'macro_categoria'] = nueva_macro
    mapeo.loc[mask, 'metodo_asignacion'] = 'correccion_manual'
    print(f"  ✓ {delito_original}")
    print(f"    {vieja_macro} → {nueva_macro} ({conteo:,} registros)")
    corregidos += 1

print(f"\nTotal corregidos: {corregidos}")

# ============================================================================
# Re-aplicar al dataset
# ============================================================================
dict_macro = dict(zip(mapeo['delito_original'], mapeo['macro_categoria']))
dict_limpio = dict(zip(mapeo['delito_original'], mapeo['delito_limpio']))

df['macro_categoria'] = df['delito'].map(dict_macro)
df['delito_limpio'] = df['delito'].map(dict_limpio)

# Verificar nulos
nulos = df['macro_categoria'].isna().sum()
print(f"\nNulos en macro_categoria después de correcciones: {nulos}")

if nulos > 0:
    print("  Delitos sin macro:")
    for d in df[df['macro_categoria'].isna()]['delito'].unique():
        print(f"    {d}")

# ============================================================================
# Guardar archivos corregidos
# ============================================================================
mapeo.to_csv("taxonomia_delitos_mapeo.csv", index=False, encoding='utf-8-sig')
df.to_csv("carpetasFGJ_fase2.csv", index=False, encoding='utf-8-sig')

print(f"\n✓ taxonomia_delitos_mapeo.csv actualizado")
print(f"✓ carpetasFGJ_fase2.csv actualizado")

# ============================================================================
# Resumen final post-corrección
# ============================================================================
print(f"\n{'='*80}")
print(f"RESUMEN FINAL POST-CORRECCIÓN")
print(f"{'='*80}")

print(f"\n  Categorías originales:  {df['delito'].nunique()}")
print(f"  Categorías limpias:     {df['delito_limpio'].nunique()}")
print(f"  Macro-categorías:       {df['macro_categoria'].nunique()}")

print(f"\n  Distribución por macro-categoría:")
dist = df['macro_categoria'].value_counts()
for macro, count in dist.items():
    pct = count/len(df)*100
    bar = '█' * int(pct)
    print(f"    {macro:45s} {count:>9,} ({pct:>5.1f}%) {bar}")

# Verificar métodos de asignación
print(f"\n  Métodos de asignación en el mapeo:")
print(mapeo['metodo_asignacion'].value_counts().to_string())

In [ ]:
print(f"Nulos en latitud: {df['latitud'].isna().sum():,} ({df['latitud'].isna().mean()*100:.1f}%)")
print(f"Nulos en longitud: {df['longitud'].isna().sum():,} ({df['longitud'].isna().mean()*100:.1f}%)")
print(f"Latitud == 0: {(df['latitud']==0).sum():,}")
print(f"Rango latitud: {df['latitud'].min():.4f} a {df['latitud'].max():.4f}")
print(f"Rango longitud: {df['longitud'].min():.4f} a {df['longitud'].max():.4f}")
print(f"\nNulos en colonia_hecho: {df['colonia_hecho'].isna().sum():,}")
print(f"Nulos en alcaldia_hecho: {df['alcaldia_hecho'].isna().sum():,}")
print(f"Colonias únicas: {df['colonia_hecho'].nunique()}")
print(f"Alcaldías únicas: {df['alcaldia_hecho'].nunique()}")
print(f"\nfecha_hecho rango: {df['fecha_hecho'].min()} a {df['fecha_hecho'].max()}")
print(f"hora_hecho_valida == True: {df['hora_hecho_valida'].sum():,} ({df['hora_hecho_valida'].mean()*100:.1f}%)")

Nulos en latitud: 101,207 (4.8%)
Nulos en longitud: 101,207 (4.8%)
Latitud == 0: 0
Rango latitud: 19.0953 a 19.5833
Rango longitud: -100.2325 a -98.9469

Nulos en colonia_hecho: 102,124
Nulos en alcaldia_hecho: 24,896
Colonias únicas: 1697
Alcaldías únicas: 17

fecha_hecho rango: 1906-06-02 a 2025-01-31 22:20:00
hora_hecho_valida == True: 2,069,920 (98.6%)


In [ ]:
# Fechas sospechosas
df['fecha_hecho_dt'] = pd.to_datetime(df['fecha_hecho'], errors='coerce')
print("Registros antes de 2016:", (df['fecha_hecho_dt'] < '2016-01-01').sum())
print("Registros 2016-2025:", ((df['fecha_hecho_dt'] >= '2016-01-01') & (df['fecha_hecho_dt'] <= '2025-12-31')).sum())
print("\nDistribución por año:")
print(df['fecha_hecho_dt'].dt.year.value_counts().sort_index().to_string())

# Coordenadas fuera de CDMX (aproximado)
fuera = ((df['latitud'] < 19.05) | (df['latitud'] > 19.60) | 
         (df['longitud'] < -99.40) | (df['longitud'] > -98.90))
print(f"\nCoordenadas fuera del bbox de CDMX: {fuera.sum():,}")

Registros antes de 2016: 32437
Registros 2016-2025: 2012191

Distribución por año:
fecha_hecho_dt
1906.0         1
1915.0         2
1917.0         1
1930.0         1
1942.0         1
1950.0         3
1952.0         1
1954.0         1
1955.0         5
1956.0         2
1957.0         2
1958.0         2
1960.0         1
1961.0         2
1962.0         7
1963.0         4
1964.0         2
1965.0         3
1966.0         6
1967.0         5
1968.0         3
1969.0        59
1970.0         7
1971.0         5
1972.0        13
1973.0         3
1974.0        16
1975.0         6
1976.0        15
1977.0         8
1978.0        10
1979.0         9
1980.0        18
1981.0        15
1982.0        18
1983.0        26
1984.0        27
1985.0        16
1986.0        20
1987.0        29
1988.0        26
1989.0        37
1990.0        48
1991.0        35
1992.0        51
1993.0        46
1994.0        44
1995.0        50
1996.0        60
1997.0        75
1998.0        86
1999.0        76
2000.0       191
2